# 프롬프트 엔지니어링

프롬프트 엔지니어링은 대규모 언어 모델이 사용자의 의도를 정확히 파악하고 최적의 결과물을 생성하도록 입력을 설계하고 최적화하는 기술이자 과학이다.

이는 단순한 질문을 넘어, AI에게 문맥, 지침, 예시를 제공하여 원하는 결과물로 유도하는 과정이다.

효과적인 프롬프트를 구성하기 위해서는 모델이 사용자의 의도를 정확히 파악하고 고품질의 결과를 생성할 수 있도록 명확한 구조를 갖추는 것이 중요하다.

### 핵심 구성 요소

- **지시** — 모델이 수행해야 할 구체적인 작업이나 명령이다. "요약하라", "분류하라", "번역하라"와 같이 명확한 동사를 사용하여 모델이 무엇을 해야 하는지 정의하며, 지시는 모호함을 피하고 구체적일수록 좋다.
- **문맥** — 모델이 더 나은 응답을 생성하도록 유도하는 배경 정보나 외부 상황이다. 모델이 상황을 추측하게 하지 말고, "이 작업은 초보자를 위한 것이다"라거나 "첨부된 재무 보고서를 바탕으로 분석하라"와 같이 필요한 정보를 충분히 제공해야 한다.
- **입력 데이터** — 모델이 처리해야 할 실제 내용이다. 요약해야 할 텍스트, 번역할 문장, 답변해야 할 질문 등이 이에 해당한다. 입력 데이터는 프롬프트 내에서 XML 태그나 특수 기호 등을 사용해 지시 사항과 명확히 구분해 주는 것이 좋다.
- **출력 지시자** — 응답의 형식이나 유형을 지정하는 요소이다. "JSON 형식으로 출력해라", "표로 만들어라", "불렛 포인트를 사용해라"와 같이 원하는 결과물의 형태를 명시한다.

### 성능 향상을 위한 추가 요소

- **역할 및 정체성** — AI에게 "당신은 노련한 데이터 과학자입니다"와 같은 페르소나를 부여한다. 이를 통해 모델의 어조, 관점, 스타일을 조정하고 특정 도메인의 전문적인 답변을 유도할 수 있다.
- **예시** — 원하는 입력과 출력의 쌍을 제공하여 모델이 패턴을 학습하게 하는 기법이다. 설명보다 예시가 더 효과적인 경우가 많다.
- **제약 사항** — 모델이 하지 말아야 할 것(부정적 제약)이나 반드시 지켜야 할 규칙(긍정적 제약)을 설정한다. 예를 들어 "전문 용어를 쓰지 마라", "500단어 이내로 작성해라" 등이 있다.
- **구조화 태그** — 프롬프트의 각 부분(지시, 문맥, 예시 등)을 명확히 구분하기 위해 XML 태그(`<context>`, `<instruction>`)나 마크다운 헤더(`#`)를 사용한다.

---

## 환경 설정

In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

---

# 핵심 프롬프트 기법

## 1. 제로샷 vs 퓨샷 프롬프팅

- **제로샷 프롬프팅** — 추가적인 예시 없이 모델에게 직접적인 지시나 질문만 제공하는 방식이다.
- **퓨샷 프롬프팅** — 모델이 패턴을 학습할 수 있도록 원하는 입력-출력 예시를 하나 이상 제공하는 기법이다.

퓨샷의 원칙:
- **다양성**: 엣지 케이스를 포함하여 다양한 예시를 제공하면 의도치 않은 패턴 학습을 방지할 수 있다.
- **일관된 형식**: 예시 간의 구조와 형식을 통일해야 모델이 혼란을 겪지 않는다.

In [6]:
# 제로샷: 예시 없이 바로 질문
prompt_zero = ChatPromptTemplate.from_messages([
    ("human", "다음 문장의 감정을 '긍정', '부정', '중립' 중 하나로 분류해: {sentence}"),
])

chain = prompt_zero | llm | parser
print("=== 제로샷 ===")
print(chain.invoke({"sentence": "이 영화 진짜 재밌었어"}))

=== 제로샷 ===
'긍정'으로 분류할 수 있습니다.


In [7]:
# 퓨샷: 예시를 제공하여 패턴 학습
prompt_few = ChatPromptTemplate.from_messages([
    ("system", "사용자의 문장을 '긍정', '부정', '중립' 중 하나로 분류해."),
    ("human", "배송이 정말 빠르고 포장도 깔끔해요"),
    ("ai", "긍정"),
    ("human", "제품이 설명과 달라서 실망했습니다"),
    ("ai", "부정"),
    ("human", "보통이에요. 그냥 쓸만합니다"),
    ("ai", "중립"),
    ("human", "{sentence}"),
])
# for i in prompt_few.messages:
#     print(type(i))

chain = prompt_few | llm | parser
print("=== 퓨샷 ===")
print(chain.invoke({"sentence": "이 영화 진짜 재밌었어"}))

=== 퓨샷 ===
긍정


같은 감정 분류 태스크지만, 퓨샷은 예시를 통해 "한 단어로만 답하라"는 패턴까지 학습시킨다.

제로샷은 장황한 설명이 붙을 수 있지만, 퓨샷은 예시의 형식을 따라 간결하게 답한다.

퓨샷의 핵심은 **예시의 패턴이 일관되어야** 한다는 것이다.

---

## 2. 역할 부여 (Role Prompting)

AI에게 특정 캐릭터나 직업적 역할을 부여하여 응답의 어조, 관점, 스타일을 조정하는 기법이다.

시스템 프롬프트에 역할을 설정하면, 모델은 해당 도메인의 전문적인 관점을 채택하여 더 깊이 있는 답변을 제공한다.
역할을 구체적으로 설정할수록 답변의 전문성과 일관성이 높아진다.

In [8]:
# 역할 없이 질문
question = "우리 회사의 레거시 시스템을 지금 리팩토링해야 할까?"

prompt_no_role = ChatPromptTemplate.from_messages([
    ("human", question),
])

chain = prompt_no_role | llm | parser
print("=== 역할 없음 ===")
print(chain.invoke({}))

=== 역할 없음 ===
레거시 시스템을 리팩토링할지 여부는 여러 가지 요소에 따라 달라집니다. 다음은 고려해야 할 몇 가지 주요 사항입니다:

1. **시스템의 상태**: 레거시 시스템이 얼마나 오래되었고, 기술적으로 얼마나 유지보수가 어려운지 평가하세요. 코드가 복잡하고 이해하기 어렵거나, 버그가 자주 발생한다면 리팩토링이 필요할 수 있습니다.

2. **비즈니스 요구사항**: 현재 비즈니스 요구사항이 변화하고 있는지, 또는 새로운 기능이 필요하다면 리팩토링이 필요할 수 있습니다. 레거시 시스템이 새로운 요구사항을 수용하기 어려운 경우, 리팩토링을 고려해야 합니다.

3. **비용과 시간**: 리팩토링에 드는 비용과 시간을 평가하세요. 리팩토링이 장기적으로 비용을 절감하고 효율성을 높일 수 있다면, 지금 시작하는 것이 좋습니다.

4. **기술 부채**: 레거시 시스템이 쌓아온 기술 부채를 해결할 필요가 있는지 검토하세요. 기술 부채가 쌓이면 시스템의 유연성과 확장성이 떨어질 수 있습니다.

5. **팀의 역량**: 팀이 새로운 기술 스택이나 아키텍처에 대한 경험이 있는지, 또는 리팩토링을 위한 충분한 리소스가 있는지 고려하세요.

6. **리스크 관리**: 리팩토링 과정에서 발생할 수 있는 리스크를 평가하고, 이를 관리할 수 있는 계획이 있는지 확인하세요.

이러한 요소들을 종합적으로 고려하여, 리팩토링이 현재 시점에서 필요한지 판단하는 것이 중요합니다. 필요하다면 전문가의 조언을 받는 것도 좋은 방법입니다.


In [9]:
# 같은 질문, 다른 역할 3개 비교
roles = {
    "CEO": "너는 시리즈B 스타트업 CEO야. 다음 분기 매출 목표가 급하고, 투자자 미팅이 잡혀 있어. 경영 관점에서 답해.",
    "CTO": "너는 이 시스템을 5년째 유지보수하고 있는 CTO야. 기술 부채가 쌓여서 신규 기능 개발 속도가 계속 느려지고 있어. 기술 관점에서 답해.",
    "현직 개발자": "너는 이 레거시 코드를 매일 다루는 3년차 개발자야. 버그 수정할 때마다 다른 곳이 터지고, 테스트도 없어. 솔직하게 답해.",
}

for role_name, role_desc in roles.items():
    prompt = ChatPromptTemplate.from_messages([
        ("system", role_desc),
        ("human", question),
    ])
    chain = prompt | llm | parser
    print(f"=== {role_name} ===")
    print(chain.invoke({}))
    print()

=== CEO ===
레거시 시스템의 리팩토링 여부는 여러 가지 요소를 고려해야 합니다. 다음은 경영 관점에서 고려해야 할 주요 사항들입니다.

1. **비용 vs. 이익 분석**: 리팩토링에 드는 비용과 시간을 평가해야 합니다. 현재 시스템의 유지보수 비용이 증가하고 있거나, 새로운 기능 추가가 어려운 상황이라면 리팩토링이 필요할 수 있습니다. 반면, 단기적으로 매출 목표가 급한 상황이라면 리팩토링에 투자하는 것이 적절한지 고민해야 합니다.

2. **비즈니스 목표와의 정렬**: 현재 매출 목표와 투자자 미팅이 중요한 시점에서, 리팩토링이 비즈니스 목표에 어떻게 기여할 수 있는지를 분석해야 합니다. 리팩토링이 매출 증가에 직접적으로 기여할 수 있는지, 아니면 장기적인 성장에 필요한 기반을 마련하는 것인지 판단해야 합니다.

3. **기술 부채**: 레거시 시스템이 기술 부채를 쌓고 있다면, 이를 해결하는 것이 중요합니다. 기술 부채가 쌓이면 향후 개발 속도와 품질에 부정적인 영향을 미칠 수 있습니다.

4. **팀의 역량**: 리팩토링을 수행할 팀의 역량과 리소스를 고려해야 합니다. 현재 팀이 리팩토링을 수행할 수 있는 기술적 능력과 시간이 있는지 평가해야 합니다.

5. **투자자와의 커뮤니케이션**: 투자자 미팅에서 레거시 시스템의 리팩토링 계획을 어떻게 설명할 것인지도 중요합니다. 리팩토링이 매출 성장에 어떻게 기여할 것인지, 그리고 이를 통해 얻을 수 있는 장기적인 이점을 명확히 전달할 수 있어야 합니다.

결론적으로, 리팩토링이 필요할 수 있지만, 현재의 매출 목표와 투자자 미팅을 고려할 때, 단기적인 성과를 우선시하는 것이 더 나은 전략일 수 있습니다. 리팩토링을 진행하기 전에 충분한 분석과 계획이 필요합니다.

=== CTO ===
레거시 시스템의 리팩토링 여부는 여러 가지 요소에 따라 결정됩니다. 다음은 기술 관점에서 고려해야 할 주요 사항들입니다:

1. **기술 부채의 정도**: 현재 시스템에서 쌓인 기술 부채가 얼마나 심각한지 평가해야 합

같은 질문이라도 역할에 따라 **결론 자체가 달라진다.**

| 역할 | 입장 | 핵심 근거 |
|------|------|----------|
| CEO | 지금은 안 돼 | 매출 목표, 투자자 일정, 리소스 부족 |
| CTO | 지금 해야 해 | 기술 부채 누적, 개발 속도 저하, 장기 비용 |
| 개발자 | 제발 해줘 | 매일 고통, 버그 연쇄, 테스트 부재 |

역할을 구체적으로 설정할수록 (상황, 맥락, 이해관계 등) 모델이 해당 입장에 몰입하여 현실적인 답변을 생성한다.

---

# 프롬프트 구조화 및 제어

## 3. 출력 형식 지정

원하는 결과물의 형식을 구체적으로 명시하면 파싱하기 쉽고 일관된 결과를 얻을 수 있다.

- "표로 만들어라", "JSON 형식으로 출력해라"와 같이 명확한 지시를 포함한다.
- "하지 마라"보다는 "하라"는 긍정적 지시가 더 효과적이다.

In [10]:
# 형식 미지정
prompt_no_format = ChatPromptTemplate.from_messages([
    ("human", "Python, JavaScript, Go를 비교해줘"),
])

chain = prompt_no_format | llm | parser
print("=== 형식 미지정 ===")
print(chain.invoke({}))

=== 형식 미지정 ===
Python, JavaScript, Go는 각각의 특성과 장점이 있는 프로그래밍 언어입니다. 이들 언어를 비교해보면 다음과 같은 주요 차이점과 특징이 있습니다.

### 1. **Python**
- **용도**: 데이터 과학, 웹 개발, 자동화, 인공지능, 머신러닝 등 다양한 분야에서 사용됩니다.
- **문법**: 간결하고 읽기 쉬운 문법을 가지고 있어 초보자에게 적합합니다.
- **생태계**: 풍부한 라이브러리와 프레임워크(예: Django, Flask, Pandas, NumPy 등)가 있어 다양한 작업을 쉽게 수행할 수 있습니다.
- **성능**: 인터프리터 언어로, 컴파일 언어에 비해 실행 속도가 느릴 수 있습니다. 그러나 C로 작성된 라이브러리를 통해 성능을 보완할 수 있습니다.
- **비동기 처리**: 비동기 프로그래밍을 지원하지만, JavaScript에 비해 상대적으로 복잡할 수 있습니다.

### 2. **JavaScript**
- **용도**: 주로 웹 개발에 사용되며, 클라이언트 사이드와 서버 사이드(예: Node.js) 모두에서 활용됩니다.
- **문법**: 유연하지만, 초보자에게는 다소 복잡할 수 있는 문법을 가지고 있습니다. 특히 비동기 처리와 콜백 패턴이 중요합니다.
- **생태계**: 방대한 라이브러리와 프레임워크(예: React, Angular, Vue.js 등)가 있어 웹 애플리케이션 개발에 매우 유용합니다.
- **성능**: JIT(Just-In-Time) 컴파일러를 사용하여 빠른 실행 속도를 제공합니다.
- **비동기 처리**: Promise, async/await 구문을 통해 비동기 프로그래밍을 쉽게 처리할 수 있습니다.

### 3. **Go (Golang)**
- **용도**: 시스템 프로그래밍, 서버 사이드 애플리케이션, 클라우드 서비스 등에서 주로 사용됩니다.
- **문법**: 간결하고 명확한 문법을 가지고 있으며, 정적 타입 언어입니다. 컴파일 언어로, 코드 작성 후 컴파일하여 실행합니다.


In [11]:
# 형식 지정: 표로 출력
prompt_table = ChatPromptTemplate.from_messages([
    ("system", "답변을 마크다운 표 형식으로 작성해. "
               "열은 '언어', '장점', '단점', '주요 사용처'로 구성해."),
    ("human", "Python, JavaScript, Go를 비교해줘"),
])

chain = prompt_table | llm | parser
print("=== 표 형식 ===")
print(chain.invoke({}))

=== 표 형식 ===
| 언어         | 장점                                         | 단점                                         | 주요 사용처                     |
|--------------|--------------------------------------------|--------------------------------------------|---------------------------------|
| Python       | - 간결하고 읽기 쉬운 문법<br>- 풍부한 라이브러리 지원<br>- 데이터 과학 및 머신러닝에 강점 | - 실행 속도가 느림<br>- 모바일 앱 개발에 적합하지 않음 | 데이터 과학, 웹 개발, 자동화, 머신러닝 |
| JavaScript   | - 웹 브라우저에서 실행 가능<br>- 비동기 프로그래밍 지원<br>- 대규모 커뮤니티와 생태계 | - 동적 타이핑으로 인한 오류 가능성<br>- 복잡한 코드 관리 어려움 | 웹 개발, 서버 사이드 개발(Node.js), 게임 개발 |
| Go           | - 높은 성능과 효율성<br>- 간결한 문법과 강력한 동시성 지원<br>- 정적 타이핑으로 안정성 높음 | - 상대적으로 적은 라이브러리<br>- 제네릭 지원 부족 (Go 1.18부터 지원) | 클라우드 서비스, 시스템 프로그래밍, 마이크로서비스 |


In [12]:
# 형식 지정: JSON으로 출력
prompt_json = ChatPromptTemplate.from_messages([
    ("system", """답변을 아래 JSON 형식으로만 작성해. 추가 설명 없이 JSON만 출력해.

[{{
  "language": "언어명",
  "pros": ["장점1", "장점2"],
  "cons": ["단점1", "단점2"],
  "use_cases": ["사용처1", "사용처2"]
}}]"""),
    ("human", "Python, JavaScript, Go를 비교해줘"),
])

chain = prompt_json | llm | parser
result = chain.invoke({})
print("=== JSON 형식 ===")
print(result)

=== JSON 형식 ===
[{
  "language": "Python",
  "pros": ["간결하고 읽기 쉬운 문법", "풍부한 라이브러리와 프레임워크"],
  "cons": ["느린 실행 속도", "모바일 개발에 적합하지 않음"],
  "use_cases": ["데이터 분석", "웹 개발", "인공지능"]
}, {
  "language": "JavaScript",
  "pros": ["브라우저에서 실행 가능", "비동기 프로그래밍 지원"],
  "cons": ["동적 타이핑으로 인한 오류 가능성", "상대적으로 복잡한 문법"],
  "use_cases": ["웹 개발", "서버 사이드 개발", "게임 개발"]
}, {
  "language": "Go",
  "pros": ["빠른 실행 속도", "간결한 문법과 강력한 동시성 지원"],
  "cons": ["제한된 라이브러리 생태계", "제네릭 지원 부족"],
  "use_cases": ["서버 개발", "클라우드 서비스", "네트워크 프로그래밍"]
}]


JSON 형식으로 받으면 후처리가 쉽다.

In [ ]:
import json

data = json.loads(result)
for lang in data:
    print(f"{lang['language']}: {', '.join(lang['pros'])}")

Python: 간결하고 읽기 쉬운 문법, 풍부한 라이브러리와 프레임워크
JavaScript: 브라우저에서 실행 가능, 비동기 프로그래밍 지원
Go: 빠른 실행 속도, 간결한 문법과 강력한 동시성 지원


> **참고:** 프롬프트로 "JSON으로 줘"라고 하면 가끔 형식이 깨질 수 있다. LangChain에서는 `llm.with_structured_output()`을 사용하면 LLM 응답을 Pydantic 객체로 바로 받을 수 있다.

In [14]:
# 제약 없음
prompt_no_constraint = ChatPromptTemplate.from_messages([
    ("human", "건강하게 오래 사는 방법을 알려줘"),
])

chain = prompt_no_constraint | llm | parser
print("=== 제약 없음 ===")
print(chain.invoke({}))

=== 제약 없음 ===
건강하게 오래 사는 방법은 여러 가지가 있습니다. 아래에 몇 가지 중요한 팁을 소개합니다:

1. **균형 잡힌 식사**: 다양한 영양소를 포함한 균형 잡힌 식사를 하세요. 과일, 채소, 통곡물, 단백질(고기, 생선, 콩류 등), 건강한 지방(올리브유, 아보카도 등)을 포함하는 것이 좋습니다.

2. **규칙적인 운동**: 주 150분 이상의 중간 강도 유산소 운동(걷기, 자전거 타기 등)과 근력 운동을 포함하세요. 운동은 심혈관 건강을 개선하고 체중을 관리하는 데 도움이 됩니다.

3. **충분한 수면**: 성인은 보통 7-9시간의 수면이 필요합니다. 규칙적인 수면 패턴을 유지하고, 수면 환경을 편안하게 만들어 주세요.

4. **스트레스 관리**: 명상, 요가, 심호흡, 취미 활동 등을 통해 스트레스를 관리하세요. 스트레스는 건강에 부정적인 영향을 미칠 수 있습니다.

5. **사회적 관계 유지**: 가족, 친구와의 관계를 소중히 여기고, 사회적 활동에 참여하세요. 긍정적인 사회적 관계는 정신 건강에 큰 도움이 됩니다.

6. **정기적인 건강 검진**: 정기적으로 건강 검진을 받고, 필요한 예방접종을 받는 것이 중요합니다. 조기 발견이 많은 질병을 예방할 수 있습니다.

7. **금연 및 절주**: 흡연은 여러 질병의 주요 원인이므로 금연하는 것이 좋습니다. 음주는 적당히 하세요.

8. **정신적 건강 관리**: 긍정적인 사고방식을 유지하고, 필요할 경우 전문가의 도움을 받는 것도 중요합니다.

이러한 방법들을 일상생활에 적용하면 건강하게 오래 사는 데 도움이 될 것입니다.


In [15]:
# 제약 조건 적용
prompt_constrained = ChatPromptTemplate.from_messages([
    ("system", """다음 규칙을 반드시 지켜:
- 3문장 이내로 답변해
- 각 문장 앞에 번호를 붙여
- 마크다운 볼드(**) 없이 plain text로만 써
- '~입니다', '~합니다' 대신 '~이다', '~한다' 체를 써
- 추상적인 조언 대신 구체적인 수치나 행동을 포함해"""),
    ("human", "건강하게 오래 사는 방법을 알려줘"),
])

chain = prompt_constrained | llm | parser
print("=== 제약 조건 적용 ===")
print(chain.invoke({}))

=== 제약 조건 적용 ===
1. 매일 30분 이상 유산소 운동을 하여 심혈관 건강을 유지한다.  
2. 하루에 최소 5인분의 과일과 채소를 섭취하여 영양소를 충분히 공급한다.  
3. 매주 7시간 이상 수면을 취하여 면역력을 높이고 스트레스를 줄인다.  


---

## 5. 구조화 태그 (XML 태그) 활용

프롬프트 내의 지시사항, 문맥, 예시를 명확히 구분하기 위해 XML 태그를 사용한다.

모델이 프롬프트의 각 부분을 명확하게 파싱하도록 도와 환각을 줄이고 정확도를 높인다.
특히 사용자 입력을 그대로 프롬프트에 넣을 때, 프롬프트 인젝션을 방지하는 효과도 있다.

In [ ]:
prompt_xml = ChatPromptTemplate.from_messages([
    ("system", """아래 <article> 태그 안의 글을 분석해서 다음을 추출해:
1. 핵심 주제 (한 줄)
2. 키워드 3개
3. 한 줄 요약

<article> 태그 밖의 내용은 무시해."""),
    ("human", """<article>
{article}
</article>"""),
])

chain = prompt_xml | llm | parser

article = """최근 AI 기술의 발전으로 소프트웨어 개발 방식이 크게 변화하고 있다.
GitHub Copilot, Cursor 같은 AI 코딩 도구가 보편화되면서
개발자의 역할이 코드 작성에서 코드 검증과 설계로 이동하고 있다.
특히 프롬프트 엔지니어링 능력이 새로운 핵심 역량으로 부상하고 있다."""

print(chain.invoke({"article": article}))

---

# 고급 전략 및 최적화

---

## 8. 연쇄적 사고 (Chain of Thought)

복잡한 추론이 필요한 작업에서 모델에게 단계별로 생각하라고 지시하여 논리적 오류를 줄이는 기법이다.

- **기본 CoT** — 프롬프트에 "단계별로 생각하라"는 문구를 추가한다.
- **구조화된 CoT** — 모델이 추론 과정과 최종 답변을 분리하도록 유도한다. 예를 들어, `<thinking>` 태그 안에서 먼저 추론하고 `<answer>` 태그에 답을 내놓게 한다.

수학 문제, 논리적 분석, 복잡한 문서 작성 시 정확도와 일관성이 향상되며, 모델의 사고 과정을 통해 디버깅이 용이해진다.

In [ ]:
# 바로 답하게 하기
prompt_direct = ChatPromptTemplate.from_messages([
    ("human", "가게에 사과가 23개 있었다. 11개를 팔고, 새로 6개를 들여왔다. "
             "다음 날 8개를 더 팔았다. 남은 사과는 몇 개인가?"),
])

chain = prompt_direct | llm | parser
print("=== 바로 답하기 ===")
print(chain.invoke({}))

In [ ]:
# 기본 CoT: 단계별 사고 유도
prompt_cot = ChatPromptTemplate.from_messages([
    ("system", "문제를 단계별로 풀어. 각 단계를 명시하고, 마지막에 최종 답을 써."),
    ("human", "가게에 사과가 23개 있었다. 11개를 팔고, 새로 6개를 들여왔다. "
             "다음 날 8개를 더 팔았다. 남은 사과는 몇 개인가?"),
])

chain = prompt_cot | llm | parser
print("=== 기본 CoT ===")
print(chain.invoke({}))

In [ ]:
# 구조화된 CoT: 추론과 답변 분리
prompt_structured_cot = ChatPromptTemplate.from_messages([
    ("system", """문제를 풀 때 다음 형식을 따라:

<thinking>
여기서 단계별로 추론한다
</thinking>

<answer>
최종 답만 간결하게 쓴다
</answer>"""),
    ("human", "회사에 직원이 150명 있다. 1분기에 20% 늘었고, "
             "2분기에 15명이 퇴사했다. 현재 직원 수는?"),
])

chain = prompt_structured_cot | llm | parser
print("=== 구조화된 CoT ===")
print(chain.invoke({}))

=== 구조화된 CoT ===
<thinking>
1분기에 직원 수가 20% 늘었으므로, 150명의 20%는 30명입니다. 따라서 1분기 후 직원 수는 150 + 30 = 180명입니다. 
2분기에 15명이 퇴사했으므로, 180명에서 15명을 빼면 현재 직원 수는 180 - 15 = 165명입니다.
</thinking>

<answer>
165명입니다. 
</answer>


: 

> **참고: 최신 모델의 내장 CoT**
>
> 최근 출시된 모델들은 CoT가 내장되어 있어 별도 지시 없이도 내부적으로 단계별 추론을 수행한다.
>
> | 모델 | 기능 |
> |------|------|
> | OpenAI `o1`, `o3` | 추론 전용 모델, 내부 추론 후 답변 생성 |
> | OpenAI `GPT-5` | o3 수준의 추론이 기본 모델에 통합 |
> | Anthropic Claude `extended thinking` | 사고 과정을 명시적으로 출력 |
> | Google Gemini `thinking mode` | 내부 추론 체인 활용 |
>
> "reasoning", "thinking", "CoT" 모두 본질은 같다 — 답변 전에 내부적으로 단계별 사고를 거치는 것이며, 벤더마다 부르는 이름만 다르다.
>
> 다만 `gpt-4o-mini` 같은 기본 모델에서는 여전히 CoT 프롬프팅이 유효하다. 추론 특화 모델은 비용이 높으므로, 간단한 작업에서는 기본 모델 + CoT 프롬프팅이 더 효율적일 수도 있다.

## 7. 프롬프트 체이닝

복잡한 작업을 한 번에 처리하려 하지 말고, 여러 개의 하위 작업으로 나누어 순차적으로 처리하는 기법이다.

각 단계의 출력을 다음 단계의 입력으로 사용한다.

```
문서 분석 → 요점 추출 → 요약 작성 → 번역
```

각 단계에 모델이 온전히 집중할 수 있어 정확도가 높아지고, 오류 발생 시 원인을 파악하기 쉽다.

In [ ]:
# 프롬프트 체이닝: 2단계로 나누어 처리

# 1단계: 주요 주제 추출
prompt_step1 = ChatPromptTemplate.from_messages([
    ("system", "주어진 텍스트에서 주요 주제 3개를 번호 목록으로 추출해."),
    ("human", "{text}"),
])

# 2단계: 추출된 주제로 요약 작성
prompt_step2 = ChatPromptTemplate.from_messages([
    ("system", "아래 주제들을 바탕으로 3줄 요약문을 작성해."),
    ("human", "{topics}"),
])

text = """인공지능 기술이 의료 분야에서 혁신을 일으키고 있다.
딥러닝 기반 영상 진단 시스템은 X-ray와 CT 스캔에서 종양을 99% 정확도로 감지한다.
자연어 처리 기술은 환자 차트를 자동으로 분석하여 의사의 업무 부담을 줄여준다.
신약 개발에서는 AI가 후보 물질 탐색 시간을 수년에서 수개월로 단축시켰다.
다만 의료 AI의 판단에 대한 책임 소재, 환자 데이터 프라이버시 등
해결해야 할 윤리적 과제도 남아 있다."""

# 1단계 실행
chain1 = prompt_step1 | llm | parser
topics = chain1.invoke({"text": text})
print("=== 1단계: 주제 추출 ===")
print(topics)

# 2단계 실행 (1단계 결과를 입력으로)
chain2 = prompt_step2 | llm | parser
summary = chain2.invoke({"topics": topics})
print("\n=== 2단계: 요약 ===")
print(summary)

---

## 9. 모델 파라미터 조정 (Temperature)

API를 사용할 경우, 파라미터 조정을 통해 결과의 다양성을 제어할 수 있다.

| 파라미터 | 낮은 값 | 높은 값 |
|---------|--------|--------|
| temperature | 결정론적, 일관된 답변 | 창의적, 다양한 답변 |
| top_p | 높은 확률 토큰만 선택 | 더 다양한 토큰 후보 |

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("human", "'사랑'을 주제로 시 한 줄을 써줘"),
])

# temperature=0: 매번 같은 결과
llm_cold = ChatOpenAI(model="gpt-4o-mini", temperature=0)
chain_cold = prompt | llm_cold | parser

print("=== temperature=0 (3번 실행) ===")
for i in range(3):
    print(f"  {i+1}: {chain_cold.invoke({})}")

# temperature=1: 매번 다른 결과
llm_hot = ChatOpenAI(model="gpt-4o-mini", temperature=1)
chain_hot = prompt | llm_hot | parser

print("\n=== temperature=1 (3번 실행) ===")
for i in range(3):
    print(f"  {i+1}: {chain_hot.invoke({})}")

- 코드 생성, 분류, 요약 등 **정확성이 중요한 작업** → `temperature=0`
- 창작, 브레인스토밍 등 **다양성이 중요한 작업** → `temperature=0.7~1.0`

---

## 11. 긴 컨텍스트 처리

대량의 문서를 처리할 때는 데이터의 위치가 중요하다.

- 긴 문서나 데이터 같은 참조 자료를 프롬프트의 **상단에 배치**하고, 그 뒤에 지시사항과 질문을 배치하면 성능이 향상된다.
- 이는 모델이 문맥을 먼저 파악한 후 지시를 수행하기 때문이다.

```
좋은 구조:  [참조 문서] → [지시사항] → [질문]
나쁜 구조:  [질문] → [참조 문서] (문서가 길면 질문을 잊을 수 있음)
```

### 실전 팁 요약

- **명확하고 구체적으로** — 모호한 표현을 피하고, 정확한 동사와 수치를 사용한다 (예: "짧게 써라" → "500단어 내외로 써라")
- **맥락 제공** — 모델이 추측하지 않도록 필요한 배경 정보, 용어 정의, 참조 문서를 충분히 제공한다
- **반복과 실험** — 처음부터 완벽한 프롬프트는 존재하지 않는다. 결과를 보며 단어를 바꾸거나 정보를 추가/삭제하며 최적화한다
- **복잡한 작업 분해** — 작업이 너무 복잡하면 단계별 지시로 나누거나 프롬프트 체이닝을 고려한다
- **보안 및 안전** — 편향되거나 유해한 콘텐츠 생성을 방지하기 위해 프롬프트 내에서 제약 조건을 명확히 설정한다

이 기법들은 LangChain뿐 아니라 어떤 LLM API를 쓰더라도 동일하게 적용된다.